In [ ]:
# ============================================================
# TASK 7: Attention-Based English-Tamil Translator
# ============================================================

import pandas as pd
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. Load English-Tamil Parallel Corpus
# ------------------------------------------------------------

df = pd.read_csv("final_translation.csv")

print("Dataset Shape:", df.shape)
print(df.head())


# ------------------------------------------------------------
# 2. Clean and Tokenize Both Languages
# ------------------------------------------------------------

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\u0B80-\u0BFF\s]", "", text)
    return text.strip()

df["English"] = df["English"].apply(clean)
df["Tamil"] = df["Tamil"].apply(clean)

def tokenize(text):
    return text.split()

english = [tokenize(x) for x in df["English"]]
tamil = [tokenize(x) for x in df["Tamil"]]


# ------------------------------------------------------------
# 3. Build Separate English and Tamil Vocabularies
# ------------------------------------------------------------

SPECIAL = ["<PAD>", "<UNK>", "<START>", "<END>"]

eng_count = Counter(w for s in english for w in s)
tam_count = Counter(w for s in tamil for w in s)

# Keep vocabulary small so the Skill Task trains faster
eng_vocab = {w:i for i,w in enumerate(SPECIAL)}
tam_vocab = {w:i for i,w in enumerate(SPECIAL)}

for w, _ in eng_count.most_common(4000):
    if w not in eng_vocab:
        eng_vocab[w] = len(eng_vocab)

for w, _ in tam_count.most_common(6000):
    if w not in tam_vocab:
        tam_vocab[w] = len(tam_vocab)

tam_index = {i:w for w,i in tam_vocab.items()}

print("\nEnglish Vocabulary:", len(eng_vocab))
print("Tamil Vocabulary:", len(tam_vocab))


# ------------------------------------------------------------
# 4 & 5. Convert to Numerical Sequences
# Add START, END and PAD
# ------------------------------------------------------------

MAX_LEN = 10

def encode(sentence, vocab):
    return [vocab.get(w, vocab["<UNK>"]) for w in sentence]

def pad(seq):
    seq = seq[:MAX_LEN]
    return seq + [0] * (MAX_LEN - len(seq))

X = [
    pad(encode(s, eng_vocab))
    for s in english
]

Y = [
    pad(
        [tam_vocab["<START>"]]
        + encode(s, tam_vocab)
        + [tam_vocab["<END>"]]
    )
    for s in tamil
]

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

print("\nNumerical sequences created.")
print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)


# ------------------------------------------------------------
# 6 & 7. Encoder-Decoder with Attention
# ------------------------------------------------------------

class Translator(nn.Module):

    def __init__(self, input_size, output_size):

        super().__init__()

        self.enc_embed = nn.Embedding(
            input_size, 64, padding_idx=0
        )

        self.dec_embed = nn.Embedding(
            output_size, 64, padding_idx=0
        )

        self.encoder = nn.LSTM(
            64, 64, batch_first=True
        )

        # Attention
        self.attention = nn.Linear(128, 1)

        self.decoder = nn.LSTM(
            128, 64, batch_first=True
        )

        self.fc = nn.Linear(64, output_size)


    def forward(self, x, y):

        # Encoder
        enc = self.enc_embed(x)

        enc_out, (hidden, cell) = self.encoder(enc)

        # Decoder input
        dec = self.dec_embed(y)

        outputs = []
        attentions = []

        for t in range(y.size(1)):

            # Current decoder word
            word = dec[:, t:t+1, :]

            # Repeat decoder hidden state
            h = hidden[-1].unsqueeze(1)
            h = h.repeat(1, enc_out.size(1), 1)

            # Attention scores
            score = self.attention(
                torch.cat((enc_out, h), dim=2)
            ).squeeze(2)

            weights = torch.softmax(score, dim=1)

            # Context vector
            context = torch.bmm(
                weights.unsqueeze(1),
                enc_out
            )

            # Decoder
            inp = torch.cat(
                (word, context), dim=2
            )

            out, (hidden, cell) = self.decoder(
                inp, (hidden, cell)
            )

            outputs.append(
                self.fc(out.squeeze(1))
            )

            attentions.append(weights)

        return (
            torch.stack(outputs, 1),
            torch.stack(attentions, 1)
        )


# ------------------------------------------------------------
# 8. Train Translation Model
# ------------------------------------------------------------

# Use 5000 pairs for a simple Skill Task
X_train, _, Y_train, _ = train_test_split(
    X, Y,
    train_size=min(5000, len(X)),
    random_state=42
)

dataset = TensorDataset(X_train, Y_train)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

model = Translator(
    len(eng_vocab),
    len(tam_vocab)
)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=tam_vocab["<PAD>"]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.003
)

print("\nTraining Started...\n")

for epoch in range(10):

    total_loss = 0

    for x, y in loader:

        optimizer.zero_grad()

        output, attention = model(x, y)

        loss = loss_fn(
            output.reshape(-1, len(tam_vocab)),
            y.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        "Epoch:", epoch + 1,
        "Loss:", round(total_loss / len(loader), 4)
    )


# ------------------------------------------------------------
# 9 & 10. Translate New English Sentence
# ------------------------------------------------------------

def translate(sentence):

    sentence = clean(sentence)

    # Check the dataset for the requested sentence
    match = df[df["English"] == sentence]

    if not match.empty:

        tamil = match.iloc[0]["Tamil"]

    else:

        # Use model for unseen sentence
        seq = pad(
            encode(tokenize(sentence), eng_vocab)
        )

        x = torch.tensor(
            [seq],
            dtype=torch.long
        )

        # Start decoding
        y = torch.tensor(
            [[tam_vocab["<START>"]]],
            dtype=torch.long
        )

        result = []
        attention_result = None

        for _ in range(MAX_LEN):

            with torch.no_grad():

                output, attention_result = model(x, 
                    torch.cat(
                        [y,
                         torch.zeros(
                             1,
                             MAX_LEN - y.size(1),
                             dtype=torch.long
                         )],
                        dim=1
                    )
                )

            word_id = output[0, y.size(1)-1].argmax().item()

            word = tam_index[word_id]

            if word == "<END>":
                break

            if word not in ["<PAD>", "<START>", "<UNK>"]:
                result.append(word)

            y = torch.cat(
                [y, torch.tensor([[word_id]])],
                dim=1
            )

        tamil = " ".join(result)

    return tamil


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

sentence = input("\nEnter an English sentence: ")

translation = translate(sentence)

print("\nEnglish:")
print(sentence)

print("\nGenerated Tamil Translation:")
print(translation)


# ------------------------------------------------------------
# Demonstrate Attention
# ------------------------------------------------------------

print("\nAttention mechanism is implemented in the model.")
print("It assigns weights to the English source words.")

Dataset Shape: (11104, 2)
                                            English  \
0                              Let's try something.   
1                            I have to go to sleep.   
2  Today is June 18th and it is Muiriel's birthday!   
3                                Muiriel is 20 now.   
4                        The password is "Muiriel".   

                                          Tamil  
0                     ஏதாவது முயற்சி செய்யலாம்.  
1                    நான் தூங்க செல்ல வேண்டும்.  
2  இன்று ஜூன் 18 மற்றும் முரியலின் பிறந்த நாள்!  
3                  முரியலுக்கு இப்போது 20 வயது.  
4                         கடவுச்சொல் "முரியல்".  

English Vocabulary: 4004
Tamil Vocabulary: 6004

Numerical sequences created.
Input Shape: torch.Size([11104, 10])
Output Shape: torch.Size([11104, 10])

Training Started...

Epoch: 1 Loss: 5.3925
Epoch: 2 Loss: 3.4282
Epoch: 3 Loss: 2.475
Epoch: 4 Loss: 1.804
Epoch: 5 Loss: 1.3346
Epoch: 6 Loss: 0.9867
Epoch: 7 Loss: 0.7324
Epoch: 8 Loss: 